<a href="https://colab.research.google.com/github/dineshaiacademy/5-day-ai-bootcamp/blob/main/Day%201%20-%20LLM%20Fundamentals/Learning/llm_fundamentals_multi_provider.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🌐 LLM Fundamentals — One Notebook, Any Provider

This notebook covers the exact same fundamentals as `llm_fundamentals_gemini.ipynb`, but instead of being locked to one provider, you pick **any one** of:

| Provider | `PROVIDER` value |
|---|---|
| OpenAI | `"openai"` |
| Anthropic (Claude) | `"anthropic"` |
| Google (Gemini) | `"gemini"` |
| DeepSeek | `"deepseek"` |

**By the end of this notebook, you will be able to:**
1. Pick a provider and securely load its API key
2. Make your first call to an LLM and read the response
3. Control model behavior with sampling parameters
4. Steer the model's behavior with a system instruction
5. Hold a multi-turn conversation using chat history
6. Stream a response token-by-token
7. Read token usage from the response

Follow the cells **in order, top to bottom**. Every step after Step 2 works identically no matter which provider you picked — you only ever change one line (`PROVIDER`).

## ✅ Prerequisites

- Python 3.9+ (already available in Colab; use your bootcamp `venv` locally)
- **Only one** API key — for whichever provider you pick below

## 🔑 Step 1 — Get an API Key and Choose Your Provider

Pick **one** provider, get its key, and store it under the matching name:

| Provider | Get your key from | Store it as |
|---|---|---|
| OpenAI | [platform.openai.com/api-keys](https://platform.openai.com/api-keys) | `OPENAI_API_KEY` |
| Anthropic (Claude) | [console.anthropic.com/settings/keys](https://console.anthropic.com/settings/keys) | `ANTHROPIC_API_KEY` |
| Google (Gemini) | [aistudio.google.com/apikey](https://aistudio.google.com/apikey) | `GAISTUDIO_API_KEY` |
| DeepSeek | [platform.deepseek.com/api_keys](https://platform.deepseek.com/api_keys) | `DEEPSEEK_API_KEY` |

Store it:
- **VS Code / local**: open the `.env` file at the repo root and set e.g. `GAISTUDIO_API_KEY=your-key-here`
- **Google Colab**: click the 🔑 key icon in the left sidebar → **Add new secret** → use the exact name from the table above → paste your key → enable notebook access

You only need **one** key — whichever provider you set `PROVIDER` to below. Never paste a key directly into a code cell.

In [ ]:
PROVIDER = "gemini"  # change to "openai", "anthropic", "gemini", or "deepseek"

PROVIDER_CONFIG = {
    "openai":    {"key_name": "OPENAI_API_KEY",    "model": "gpt-4o-mini"},
    "anthropic": {"key_name": "ANTHROPIC_API_KEY",  "model": "claude-opus-5"},
    "gemini":    {"key_name": "GAISTUDIO_API_KEY",  "model": "gemini-3.6-flash"},
    "deepseek":  {"key_name": "DEEPSEEK_API_KEY",   "model": "deepseek-chat"},
}

assert PROVIDER in PROVIDER_CONFIG, f"Unknown PROVIDER: {PROVIDER!r}"
MODEL = PROVIDER_CONFIG[PROVIDER]["model"]
print(f"Selected provider: {PROVIDER} (model: {MODEL})")

## ⚙️ Step 2 — Install SDKs and Load the Key

We install all four SDKs so this notebook runs unmodified no matter which `PROVIDER` you picked.

In [ ]:
!pip install -q openai anthropic google-genai python-dotenv

In [ ]:
import os

def get_secret(key_name: str) -> str:
    try:
        from google.colab import userdata
        return userdata.get(key_name)
    except ImportError:
        from dotenv import load_dotenv, find_dotenv
        load_dotenv(find_dotenv())
        return os.getenv(key_name)

API_KEY = get_secret(PROVIDER_CONFIG[PROVIDER]["key_name"])

if not API_KEY:
    raise ValueError(
        f"{PROVIDER_CONFIG[PROVIDER]['key_name']} not found. "
        f"Set it in Colab Secrets or in your local .env file."
    )

print(f"✅ API key loaded for provider: {PROVIDER}")

In [ ]:
if PROVIDER == "openai":
    from openai import OpenAI
    client = OpenAI(api_key=API_KEY)

elif PROVIDER == "deepseek":
    from openai import OpenAI
    client = OpenAI(api_key=API_KEY, base_url="https://api.deepseek.com")

elif PROVIDER == "anthropic":
    import anthropic
    client = anthropic.Anthropic(api_key=API_KEY)

elif PROVIDER == "gemini":
    from google import genai
    client = genai.Client(api_key=API_KEY)

print(f"✅ Connected. provider={PROVIDER} model={MODEL}")

## 🔧 Step 3 — One `generate()` Function for All Four Providers

Every provider takes a prompt in and returns text out, but the exact call shape differs. `generate()` hides that behind one interface: `generate(prompt, system=None, temperature=None, max_tokens=500)` → `(text, usage)`.

> ⚠️ **If a call ever returns empty text**, some models "think" before answering and that thinking counts against `max_tokens`. Raise `max_tokens` rather than assuming the call failed.

In [ ]:
def generate(prompt: str, system: str = None, temperature: float = None, max_tokens: int = 500):
    """Returns (text, usage_dict). Same call, any provider."""

    if PROVIDER in ("openai", "deepseek"):
        messages = []
        if system:
            messages.append({"role": "system", "content": system})
        messages.append({"role": "user", "content": prompt})

        kwargs = {"model": MODEL, "messages": messages, "max_tokens": max_tokens}
        if temperature is not None:
            kwargs["temperature"] = temperature

        response = client.chat.completions.create(**kwargs)
        text = response.choices[0].message.content
        usage = {
            "input_tokens": response.usage.prompt_tokens,
            "output_tokens": response.usage.completion_tokens,
        }
        return text, usage

    if PROVIDER == "anthropic":
        # Claude's current models don't accept temperature at all — adaptive
        # thinking replaces it (see Step 5). We simply don't send it.
        kwargs = {
            "model": MODEL,
            "max_tokens": max_tokens,
            "messages": [{"role": "user", "content": prompt}],
        }
        if system:
            kwargs["system"] = system

        response = client.messages.create(**kwargs)
        text = next((b.text for b in response.content if b.type == "text"), None)
        usage = {
            "input_tokens": response.usage.input_tokens,
            "output_tokens": response.usage.output_tokens,
        }
        return text, usage

    if PROVIDER == "gemini":
        from google.genai import types

        config_kwargs = {"max_output_tokens": max_tokens}
        if temperature is not None:
            config_kwargs["temperature"] = temperature
        if system:
            config_kwargs["system_instruction"] = system

        response = client.models.generate_content(
            model=MODEL,
            contents=prompt,
            config=types.GenerateContentConfig(**config_kwargs),
        )
        usage = {
            "input_tokens": response.usage_metadata.prompt_token_count,
            "output_tokens": response.usage_metadata.candidates_token_count,
        }
        return response.text, usage

    raise ValueError(f"Unknown provider: {PROVIDER}")

## 💬 Step 4 — Your First LLM Call

At its core, an LLM takes text in (a **prompt**) and returns text out (a **completion**) — predicted one token at a time based on patterns learned from training data.

In [ ]:
text, usage = generate("Explain what a Large Language Model is, in exactly two sentences.")
print(text)

## 🎛️ Step 5 — Controlling Behavior with Parameters

For **OpenAI, Gemini, and DeepSeek**, `temperature` controls randomness — low = focused/deterministic, high = creative/varied (typical range 0.0–2.0).

**Claude's current models (`claude-opus-5`) don't expose `temperature` at all** — they "think" adaptively by default, and creativity/depth is instead controlled with `output_config={"effort": ...}` (`low` → `max`). The cell below runs the provider-appropriate comparison automatically.

In [ ]:
prompt = "Give me one creative name for a coffee shop."

if PROVIDER == "anthropic":
    for effort in ["low", "high"]:
        response = client.messages.create(
            model=MODEL,
            max_tokens=500,
            output_config={"effort": effort},
            messages=[{"role": "user", "content": prompt}],
        )
        text = next((b.text for b in response.content if b.type == "text"), None)
        print(f"effort={effort} -> {text.strip()}")
else:
    for temp in [0.0, 1.5]:
        text, _ = generate(prompt, temperature=temp)
        print(f"temperature={temp} -> {text.strip()}")

## 🧭 Step 6 — System Instructions

A **system instruction** sets the model's persona and ground rules before it sees the user's prompt — it's how you steer tone, role, and constraints consistently across every call.

In [ ]:
text, _ = generate(
    "How do I center a div?",
    system="You are a terse senior engineer. Answer in at most 2 lines, no fluff.",
)
print(text)

## 🔄 Step 7 — Multi-Turn Conversations

LLMs are **stateless** — each call has no memory of previous ones. A chat "remembers" only because the full conversation history is re-sent with every request. `Conversation` below hides that bookkeeping (Gemini's SDK manages history internally via a chat session; the other three need the message list resent manually).

In [ ]:
class Conversation:
    def __init__(self):
        if PROVIDER == "gemini":
            self._chat = client.chats.create(model=MODEL)
        else:
            self._history = []

    def send(self, user_message: str) -> str:
        if PROVIDER == "gemini":
            return self._chat.send_message(user_message).text

        self._history.append({"role": "user", "content": user_message})

        if PROVIDER in ("openai", "deepseek"):
            response = client.chat.completions.create(
                model=MODEL, messages=self._history, max_tokens=500,
            )
            text = response.choices[0].message.content
        else:  # anthropic
            response = client.messages.create(
                model=MODEL, max_tokens=500, messages=self._history,
            )
            text = next((b.text for b in response.content if b.type == "text"), None)

        self._history.append({"role": "assistant", "content": text})
        return text


conversation = Conversation()
print("Model:", conversation.send("My name is Dinesh and I'm building an AI bootcamp."))
print("Model:", conversation.send("What did I say I'm building?"))

## ⚡ Step 8 — Streaming Responses

Instead of waiting for the entire response, you can stream it token-by-token — this is what powers the "typing" effect in apps like ChatGPT, Claude, and Gemini.

In [ ]:
prompt = "Count from 1 to 5, one number per line."

if PROVIDER in ("openai", "deepseek"):
    for chunk in client.chat.completions.create(
        model=MODEL, messages=[{"role": "user", "content": prompt}],
        max_tokens=500, stream=True,
    ):
        delta = chunk.choices[0].delta.content
        if delta:
            print(delta, end="", flush=True)

elif PROVIDER == "anthropic":
    with client.messages.stream(
        model=MODEL, max_tokens=500,
        messages=[{"role": "user", "content": prompt}],
    ) as stream:
        for text in stream.text_stream:
            print(text, end="", flush=True)

elif PROVIDER == "gemini":
    for chunk in client.models.generate_content_stream(model=MODEL, contents=prompt):
        if chunk.text:
            print(chunk.text, end="", flush=True)

## 🔢 Step 9 — Tokens and Usage

LLMs don't read words — they read **tokens** (roughly 4 characters of English text each), and every provider reports how many tokens a call used. `generate()` already returns this as its second value.

In [ ]:
text, usage = generate("The quick brown fox jumps over the lazy dog.", max_tokens=200)
print(f"Response: {text.strip()!r}")
print(f"Usage: {usage}")

## 🎯 Recap

You've now covered the core mechanics behind every LLM-powered app — and, more importantly, seen exactly where providers agree and where they don't:

| Concept | What you learned |
|---|---|
| Secure key handling | `.env` locally, Colab Secrets in the cloud — same pattern, different key name per provider |
| Basic call | prompt in → completion out, same shape everywhere |
| Sampling controls | `temperature` on OpenAI/Gemini/DeepSeek; `effort` on current Claude models |
| System instructions | steering tone and role |
| Multi-turn chat | history re-sent on every call (or managed for you, as with Gemini's chat session) |
| Streaming | token-by-token delivery, one SDK call shape per provider |
| Tokens & usage | every provider reports input/output token counts on the response |

**Next:** head to `Day 1 - LLM Fundamentals/Projects/` to apply these concepts in a real Streamlit app, or copy a starter from `Templates/`. Try switching `PROVIDER` at the top of this notebook and re-running — everything from Step 4 onward should work unchanged.